# مشروع توقع أسعار العقارات باستخدام تعلم الآلة 🏠📊
مرحباً بك في هذا المشروع! الهدف هنا هو بناء وتدريب نموذج ذكاء اصطناعي قادر على توقع أسعار العقارات بناءً على مواصفاتها الأساسية (المساحة، عدد الغرف، وعدد دورات المياه).

---
## 1. استيراد المكتبات الأساسية
في هذه الخطوة، نقوم باستيراد المكتبات اللازمة للتعامل مع البيانات وبناء النموذج.

In [4]:
import pandas as pd           
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

## 2. تحميل البيانات وتجهيز النموذج
هنا نقوم بالخطوات التالية:
1. قراءة ملف البيانات `Aqar_data.csv` وتحديد الأعمدة الرقمية المهمة فقط.
2. فصل المتغيرات المستقلة (المميزات `x`) عن المتغير التابع (الهدف `y` وهو السعر).
3. تعريف نموذج **Random Forest Regressor** مع ضبط المعلمات (Hyperparameters) للحد من مشكلة الـ *Overfitting*.

In [11]:
df = pd.read_csv("Aqar_data.csv")
df['neighborhood'] = df['location'].str.replace('حي', '').str.replace('- الرياض', '').str.strip()


## 3. تقسيم البيانات وتقييم أداء النموذج
نقوم بتقسيم البيانات إلى مجموعتي تدريب واختبار بنسبة (80:20)، ثم ندرب النموذج ونحسب دقة التوقع ($R^2\ Score$) للتأكد من توازن الأداء وعدم وجود حفظ صم للبيانات (*Overfitting*).

In [ ]:


df['listTitle'] = df['listTitle'].fillna('')
df['details'] = df['details'].fillna('')

df['is_villa'] = (
    df['listTitle'].str.contains('فيلا', na=False)
    | df['details'].str.contains('فيلا', na=False)
).astype(int)
df['is_apartment'] = (
    df['listTitle'].str.contains('شقة', na=False)
    | df['details'].str.contains('شقة', na=False)
).astype(int)
df['has_driver_room'] = df['details'].str.contains('سائق', na=False).astype(int)
df['has_maid_room'] = df['details'].str.contains('خادمة', na=False).astype(int)
df['is_new'] = df['details'].str.contains('جديد', na=False).astype(int)


clean_df = df[
    (df['price'] < df['price'].quantile(0.98))
    & (df['size'] < df['size'].quantile(0.98))
].copy()

# 3. تحويل السعر لوغاريتمياً
X = clean_df.drop(columns=['price'])
y = np.log1p(clean_df['price'])

# 4. تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train = X_train.copy()
X_test = X_test.copy()

# 5. Target Encoding للأحياء
m = 15
global_mean = y_train.mean()
stats = y_train.groupby(X_train['neighborhood']).agg(['count', 'mean'])
smooth = (stats['count'] * stats['mean'] + m * global_mean) / (
    stats['count'] + m
)

X_train['neighborhood_encoded'] = (
    X_train['neighborhood'].map(smooth).fillna(global_mean)
)
X_test['neighborhood_encoded'] = (
    X_test['neighborhood'].map(smooth).fillna(global_mean)
)

# قائمة الميزات كاملة بعد الإضافة
features = [
    'size',
    'bedrooms',
    'bathrooms',
    'neighborhood_encoded',
    'is_villa',
    'is_apartment',
    'has_driver_room',
    'has_maid_room',
    'is_new',
]

# 6. تدريب المودل
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,  
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
)

model.fit(X_train[features], y_train)

train_score = model.score(X_train[features], y_train)
test_score = model.score(X_test[features], y_test)

print(f'دقة التدريب: {train_score:.4f}')
print(f'دقة الاختبار: {test_score:.4f}')

دقة التدريب: 0.8212
دقة الاختبار: 0.7657


## 4. نظام التوقع التفاعلي مع المستخدم 
في هذه الخطوة الأخيرة، نقوم بتدريب النموذج على كامل البيانات المتاحة لرفع دقة التوقع، ثم نتيح للمستخدم إدخال مواصفات العقار الذي يرغب في معرفة سعره بشكل تفاعلي ليقوم النموذج بحسابه فوراً.

In [ ]:
# 1. تجهيز المودل النهائي على كامل البيانات
m = 15
y_log_all = np.log1p(clean_df['price'])
global_mean_all = y_log_all.mean()

stats_all = y_log_all.groupby(clean_df['neighborhood']).agg(['count', 'mean'])
smooth_all = (stats_all['count'] * stats_all['mean'] + m * global_mean_all) / (
    stats_all['count'] + m
)

clean_df_encoded = clean_df.copy()
clean_df_encoded['neighborhood_encoded'] = (
    clean_df_encoded['neighborhood'].map(smooth_all).fillna(global_mean_all)
)

X_full = clean_df_encoded[features]
y_full = y_log_all

final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
)
final_model.fit(X_full, y_full)

print('---------------- تم تدريب النموذج بنجاح ----------------')

# 2. إدخال بيانات العقار التفصيلية
user_size = float(input('أدخل المساحة (م²): '))
user_beds = float(input('أدخل عدد غرف النوم: '))
user_baths = float(input('أدخل عدد دورات المياه: '))
user_nh = input('أدخل اسم الحي: ')

prop_type = input('نوع العقار (فيلا / شقة / غير ذلك): ').strip()
is_villa = 1 if 'فيلا' in prop_type else 0
is_apt = 1 if 'شقة' in prop_type else 0

has_driver = int(
    input('هل يوجد غرفة سائق؟ (1 نعم / 0 لا): ') or 0
)
has_maid = int(
    input('هل يوجد غرفة خادمة؟ (1 نعم / 0 لا): ') or 0
)
is_new = int(
    input('هل العقار جديد؟ (1 نعم / 0 لا): ') or 0
)

user_encoded_nh = smooth_all.get(user_nh, global_mean_all)

# تجهيز الإدخال بنفس ترتيب الميزات
user_input = pd.DataFrame(
    [[
        user_size,
        user_beds,
        user_baths,
        user_encoded_nh,
        is_villa,
        is_apt,
        has_driver,
        has_maid,
        is_new,
    ]],
    columns=features,
)

# 3. التوقع
predict_log = final_model.predict(user_input)
predict_price = np.expm1(predict_log)[0]

print('-' * 40)
print(f'السعر المتوقع للعقار هو: {predict_price:,.2f} ريال')
print('تنويه: السعر مجرد توقع من النموذج قد يصيب ويخطئ')
print('-' * 40)



---------------- تم تدريب النموذج بنجاح ----------------
----------------------------------------
السعر المتوقع للعقار هو: 1,212,148.62 ريال
تنويه: السعر مجرد توقع من النموذج قد يصيب ويخطئ
----------------------------------------
